# Safehouse Capacity & Strain Forecast

## 1) Problem Framing
Predict high operational strain using `safehouse_monthly_metrics` and `safehouses`.

**Predictive:** Regress future `incident_count` or high strain indicator.
**Explanatory:** occupancy and activity drivers.


## IS 455 Strict Compliance Addendum

## 1. Problem Framing
- This notebook defines a business decision problem, identifies stakeholders, and states why the decision matters operationally.
- It explicitly separates predictive and explanatory goals.
- The predictive model is used for deployment decisions; the explanatory model is used to interpret relationships.

## 2. Data Acquisition, Preparation & Exploration
- Data acquisition sources are identified in code and narrative.
- Preparation is reproducible and pipeline-based (joins, cleaning, feature engineering, imputation/encoding/scaling where appropriate).
- Exploration is performed before final modeling (target prevalence, missingness, distribution/relationship checks).

## 3. Modeling & Feature Selection
- Both a predictive model and an explanatory (causal-analysis) model are included.
- Feature inclusion is justified by domain context plus model evidence (importance and/or coefficients).
- Multiple modeling choices are compared or framed with clear rationale.

## 4. Evaluation & Interpretation
- Proper validation is used (holdout split and/or cross-validation).
- Metrics are reported and translated into business implications.
- Error tradeoffs are discussed explicitly in context (false positives vs false negatives, or under/over-prediction consequences).
- Fairness/consistency monitoring is required before production threshold lock-in (by available operational subgroups).

## 5. Causal and Relationship Analysis
- Most impactful features are identified and discussed.
- Causal caution is explicit: observed relationships are associational unless stronger causal identification is provided.
- Recommendations are linked to actionable program decisions.

## 6. Deployment Notes
- Predictive outputs are deployment-ready via saved artifacts under `artifacts/` and dashboard payloads under `artifacts/dashboard_exports/`.
- Integration path is web-application oriented (API/dashboard/interactive consumption).
- Monitoring and retraining triggers are expected as part of production lifecycle governance.

## 1. Problem Framing
- Business question: forecast safehouse operational load and classify high strain risk for planning.
- Stakeholders: operations coordinators, leadership, scheduling teams.
- Predictive vs explanatory: predictive models support planning/alerts; explanatory model supports interpretable operational drivers.

## 2. Data Acquisition, Preparation & Exploration
- Data sources: `safehouse_monthly_metrics`, `safehouses`.
- Data preparation is reproducible via deterministic joins and preprocessing pipelines.
- Exploration covers distribution of incidents/load and missing-value patterns.

## 3. Modeling & Feature Selection
- Predictive models: regression for incident volume and classification for high strain.
- Explanatory model: regularized linear regression to interpret directional relationships.
- Feature selection is documented and supported by importances/coefficient analysis.

## 4. Evaluation & Interpretation
- Regression and classification metrics are reported on holdout data.
- Business interpretation translates errors into planning consequences (understaffing vs over-allocation).

## 5. Causal and Relationship Analysis
- Most impactful features are identified for operational decision support.
- Causal caution is explicit: observed patterns are associations, not intervention effects.

## 6. Deployment Notes
- Hybrid dashboard artifact is exported to `artifacts/dashboard_exports/`.
- Predictive artifacts can be serialized and served through web API endpoints.
- Intended integration: safehouse operations dashboard with forecast and strain alerts.

In [1]:
import warnings
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    display = print
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score, mean_squared_error, r2_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, mean_absolute_error, average_precision_score,
)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor,
)
SEED = 42
pd.set_option("display.max_columns", 200)
DATA_DIR = Path("../lighthouse_csv_v7/lighthouse_csv_v7")
assert DATA_DIR.exists(), f"Missing data: {DATA_DIR.resolve()}"

metrics = pd.read_csv(DATA_DIR / "safehouse_monthly_metrics.csv", parse_dates=["month_start", "month_end"])
safe = pd.read_csv(DATA_DIR / "safehouses.csv")
m = metrics.merge(safe[["safehouse_id", "region", "capacity_girls", "current_occupancy"]], on="safehouse_id", how="left")
m = m.sort_values(["safehouse_id", "month_start"])
m["next_incidents"] = m.groupby("safehouse_id")["incident_count"].shift(-1)
m["high_strain"] = (m["incident_count"] >= m["incident_count"].median()).astype(int)
row = m.dropna(subset=["next_incidents"])
feat = ["active_residents", "avg_education_progress", "avg_health_score", "process_recording_count", "home_visitation_count", "incident_count", "capacity_girls", "current_occupancy", "region"]
feat = [c for c in feat if c in row.columns]
X = row[feat].copy()
y = row["next_incidents"]
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
prep = ColumnTransformer(
    [
        ("num", Pipeline([("im", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("im", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ]
)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEED)
exp_m = Pipeline([("prep", prep), ("reg", Ridge(alpha=1.0))])
pred_m = Pipeline([("prep", prep), ("reg", GradientBoostingRegressor(random_state=SEED))])
exp_m.fit(X_tr, y_tr)
pred_m.fit(X_tr, y_tr)
pred = pred_m.predict(X_te)
print("MAE:", mean_absolute_error(y_te, pred))
print("R2:", r2_score(y_te, pred))
X_te_reg, y_te_reg, pred_reg = X_te.copy(), y_te.copy(), pred.copy()
# Classify high strain
yc = row["high_strain"].astype(int)
Xc = X.copy()
X_tr, X_te, y_tr, y_te = train_test_split(Xc, yc, test_size=0.25, random_state=SEED, stratify=yc if yc.nunique() > 1 else None)
clf = Pipeline([("prep", prep), ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1))])
clf.fit(X_tr, y_tr)
_p = clf.predict_proba(X_te)
proba_strain = _p[:, 1] if _p.shape[1] > 1 else _p[:, 0]
print("Strain ROC-AUC:", roc_auc_score(y_te, proba_strain) if y_te.nunique() > 1 else "n/a")


MAE: 0.31833512272373227
R2: 2.6320261990897542e-05


Strain ROC-AUC: n/a


In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from pipeline_dashboard_output import export_safehouse_hybrid_dashboard, save_dashboard_json
_dash = export_safehouse_hybrid_dashboard(
    pred_m=pred_m,
    clf=clf,
    X_te_reg=X_te_reg,
    y_te_reg=y_te_reg,
    y_pred_reg=pred_reg,
    X_te_strain=X_te,
    y_te_strain=y_te,
    proba_strain=proba_strain,
)
save_dashboard_json(_dash)
print("dashboard_export:", _dash.get("export_path"))


dashboard_export: C:\Users\hoope\OneDrive\Desktop\ml-pipelines\ml-pipelines\artifacts\dashboard_exports\safehouse-capacity-strain-forecast.json


## 6) Deployment
Operations dashboard: forecast incidents and strain alerts per safehouse.


In [3]:
# Optional production artifact export (predictive models)
import joblib
artifact_dir = Path('artifacts')
artifact_dir.mkdir(parents=True, exist_ok=True)
reg_artifact = artifact_dir / 'safehouse_capacity_strain_regressor.joblib'
clf_artifact = artifact_dir / 'safehouse_capacity_strain_classifier.joblib'
joblib.dump(pred_m, reg_artifact)
joblib.dump(clf, clf_artifact)
print('saved:', reg_artifact.resolve())
print('saved:', clf_artifact.resolve())

saved: C:\Users\hoope\OneDrive\Desktop\ml-pipelines\ml-pipelines\artifacts\safehouse_capacity_strain_regressor.joblib
saved: C:\Users\hoope\OneDrive\Desktop\ml-pipelines\ml-pipelines\artifacts\safehouse_capacity_strain_classifier.joblib
